# CE541E08 — Unit 4 · Day 28 — Introduction to Pandas: Series and DataFrame

| | |
|---|---|
| **Course** | CE541E08 |
| **Department** | Civil Engineering · Christ University |
| **Instructor** | Dr. Arpan Pradhan |
| **Unit** | Unit 4 — The Pandas Library |
| **Session** | Day 28 of 45 |
| **CO** | CO4 |
| **Topics** | Why Pandas · pd.Series · pd.DataFrame · pd.read_csv · .head() · .describe() |

---
> Read the explanation before each code block. Check the expected output. Run the cell and verify. Then try the small challenge at the end.
---

In [ ]:
student_name = "Your Full Name"
roll_number  = "2024XXXXXX"
session      = "Day 28"
print(f"CE541E08 | {student_name} | {roll_number} | {session}")

---
## Section 1 — Why Pandas?

NumPy arrays are powerful for numerical computation, but they have limitations for real engineering data:

- They cannot store mixed types (numbers and text) in the same array
- They have no column names — you must remember that column 3 is "Flow" and column 4 is "Flag"
- They have no date-aware indexing
- Loading a CSV requires `np.loadtxt` with careful skiprows and dtype handling

**Pandas solves all of this.** It is built on top of NumPy and adds:

| What Pandas adds | Why it matters |
|---|---|
| Named columns | `df['Flow_m3s']` instead of `data[:,1]` |
| Row labels (index) | `df.loc['2024-07-05']` to select a date |
| Mixed types in one table | Numbers, text, and dates in the same DataFrame |
| One-line CSV loading | `pd.read_csv('cwc_data.csv')` |
| Built-in statistics | `.describe()`, `.value_counts()`, `.groupby()` |

Pandas has two core objects:
- **Series** — a 1-D labelled array (one column with an index)
- **DataFrame** — a 2-D labelled table (rows × columns, like a spreadsheet)

---
## Code Block 1 — pd.Series: Labelled 1-D Data

### What this code does

We create a Pandas Series of 12 monthly rainfall values with month-name labels as the index. We then use Series methods to compute totals, find the wettest month by name, and extract the monsoon sub-series using label-based indexing.

### Why each step is taken

**`pd.Series([...], index=[...], name='...')`:**
The `index` gives a label to each value — here the month abbreviation. `name` gives the Series itself a name (used as the column header when the Series becomes part of a DataFrame). Without an index, Pandas uses 0, 1, 2, ... — which is less meaningful.

**`.sum()` and `.max()`:**
Same as NumPy — but these are Pandas Series methods that return a scalar. They work the same way.

**`.idxmax()`:**
Returns the **label** (index value) of the maximum element — 'Jun' in this case. This is more useful than `.argmax()` which returns an integer position. With named indices, the label is directly interpretable.

**`monthly_rain[['Jun','Jul','Aug','Sep']]` — label indexing:**
Using a list of labels selects those elements by name. No need to remember that June is index 5. This is clearer, less error-prone, and self-documenting.

### Algorithm

```
1. Create Series with 12 values + month-name index

2. .sum()    → total annual rainfall
   .max()    → highest monthly value
   .idxmax() → month name with highest value

3. ['Jun','Jul','Aug','Sep']
   → select 4 monsoon months by label
   .sum() → monsoon total
```

### Expected output

```
Jan     8
Feb    12
Mar    18
Apr    52
May    87
Jun    134
Jul    118
Aug    113
Sep     95
Oct     71
Nov     44
Dec     13
Name: Rainfall_mm, dtype: int64

Total    : 765 mm
Max month: Jun (134 mm)
Monsoon  : 460 mm
```

In [ ]:
import pandas as pd

# pd.Series: 1-D labelled array
# index assigns a name to each value; name labels the Series itself
monthly_rain = pd.Series(
    [8, 12, 18, 52, 87, 134, 118, 113, 95, 71, 44, 13],
    index=['Jan','Feb','Mar','Apr','May','Jun',
           'Jul','Aug','Sep','Oct','Nov','Dec'],
    name='Rainfall_mm'
)

print(monthly_rain)
print()

print(f"Total    : {monthly_rain.sum()} mm")

# .idxmax() returns the INDEX LABEL of the maximum — 'Jun', not integer 5
print(f"Max month: {monthly_rain.idxmax()} ({monthly_rain.max()} mm)")

# Label-based selection: pass a list of index labels
# No need to remember that June is position 5
monsoon = monthly_rain[['Jun','Jul','Aug','Sep']]
print(f"Monsoon  : {monsoon.sum()} mm")

### 🔁 Try this

Find the **driest month** using `.idxmin()`.

Then compute the fraction of annual rainfall that falls in the monsoon:
`monsoon.sum() / monthly_rain.sum() * 100`

---
## Code Block 2 — pd.DataFrame: Labelled 2-D Table

### What this code does

We create a Pandas DataFrame from a Python dictionary — each key becomes a column name, each list of values becomes a column. This is how most DataFrames are created from scratch in practice.

### Why each step is taken

**Dictionary → DataFrame:**
Each key-value pair in the dictionary becomes one column. All lists must be the same length. Pandas infers the data type of each column automatically — integers, floats, and strings handled correctly.

**`df.shape`:**
Returns `(rows, columns)` — same as NumPy. Quick sanity check that the DataFrame has the expected dimensions.

**`df.dtypes`:**
Shows the inferred type for each column. `int64` for integers, `float64` for decimals, `object` for strings (Pandas stores strings as `object`). Understanding dtypes matters because some operations only work on numeric columns.

**Why a DataFrame instead of a dictionary:**
A Python dictionary does not support column-wise arithmetic, filtering rows by condition, or merging with other datasets. A DataFrame does all of this.

### Algorithm

```
1. Create a Python dictionary with 5 keys (columns)

2. pd.DataFrame(data) converts dictionary to table
   → 5 rows × 5 columns

3. print(df)       → show the full table
   df.shape        → (5, 5)
   df.dtypes       → data type per column
```

### Expected output

```
      Station    Lat    Lon  Annual_mm  Elevation_m
0   Bengaluru  12.97  77.59        967          920
1      Mysuru  12.29  76.64        784          763
2   Mangaluru  12.87  74.88       3715           22
3     Hubballi  15.36  75.12        652          678
4    Belagavi  15.85  74.50       1089          747

Shape  : (5, 5)
Dtypes :
Station        object
Lat           float64
Lon           float64
Annual_mm       int64
Elevation_m     int64
```

In [ ]:
import pandas as pd

# Create DataFrame from a dictionary
# Keys = column names; Values = lists of equal length
data = {
    'Station'    : ['Bengaluru','Mysuru','Mangaluru','Hubballi','Belagavi'],
    'Lat'        : [12.97, 12.29, 12.87, 15.36, 15.85],
    'Lon'        : [77.59, 76.64, 74.88, 75.12, 74.50],
    'Annual_mm'  : [967, 784, 3715, 652, 1089],
    'Elevation_m': [920, 763, 22, 678, 747],
}

df = pd.DataFrame(data)
print(df)
print()

# shape works the same as NumPy: (rows, columns)
print(f"Shape  : {df.shape}")

# dtypes: auto-detected per column
# object = string; int64 = integer; float64 = decimal
print(f"Dtypes :"); print(df.dtypes)

### 🔁 Try this

Add a new column `Monsoon_mm` with values `[721, 580, 3100, 490, 810]`.

Then compute `Monsoon_pct = Monsoon_mm / Annual_mm * 100` and print the station with the highest monsoon percentage using `df['Monsoon_pct'].idxmax()`.

---
## Code Block 3 — pd.read_csv: Loading Real Data

### What this code does

We simulate loading a CWC streamflow CSV file using `pd.read_csv`. In practice this would be `pd.read_csv('cwc_flow.csv')`. We use `io.StringIO` here to create an in-memory CSV string so you do not need an actual file.

### Why each step is taken

**`io.StringIO("\n".join(lines))`:**
This converts a Python list of strings into a file-like object that `pd.read_csv` can read. In real use, you would simply pass the filename: `pd.read_csv('cwc_flow.csv')`.

**`pd.read_csv` automatic handling:**
- Reads the first row as column names
- Infers data types automatically (numbers become float64, text stays object)
- Creates a default integer index (0, 1, 2, ...)

**`df.head(3)`:**
Shows the first 3 rows — standard first step after loading any dataset to verify it loaded correctly.

**`df.describe()`:**
Computes count, mean, std, min, 25th percentile, median, 75th percentile, and max for all numeric columns. One command gives a complete statistical summary.

### Algorithm

```
1. Simulate CSV data as a list of strings
   io.StringIO joins them and makes a file-like object

2. pd.read_csv(...) → DataFrame with auto-detected types

3. print(df)        → show full table
   df.dtypes        → check inferred types

4. df.head(3)       → first 3 rows
   df['Flow_m3s'].describe() → statistics for flow column

5. Boolean filtering:
   df['Flag']=='GOOD' → True/False Series
   .sum()             → count of GOOD records
```

### Expected output

```
         Date Station  Flow_m3s     Flag
0  2024-07-01     KRS     234.5     GOOD
1  2024-07-02     KRS     267.8     GOOD
...
9  2024-07-10     KRS     345.6     GOOD

GOOD records   : 7
MISSING records: 1
```

In [ ]:
import pandas as pd
import io

# Simulated CWC CSV data — in practice: pd.read_csv('cwc_flow.csv')
lines = [
    "Date,Station,Flow_m3s,Flag",
    "2024-07-01,KRS,234.5,GOOD",
    "2024-07-02,KRS,267.8,GOOD",
    "2024-07-03,KRS,-999.0,MISSING",
    "2024-07-04,KRS,890.2,GOOD",
    "2024-07-05,KRS,1245.6,HIGH",
    "2024-07-06,KRS,987.3,GOOD",
    "2024-07-07,KRS,756.4,GOOD",
    "2024-07-08,KRS,543.2,GOOD",
    "2024-07-09,KRS,412.8,SUSPECT",
    "2024-07-10,KRS,345.6,GOOD",
]

# io.StringIO creates an in-memory file — same interface as a real file
df = pd.read_csv(io.StringIO("
".join(lines)))
print(df)
print()
print("dtypes:"); print(df.dtypes)

In [ ]:
# Inspection methods — run these after loading any new dataset
print("head(3):"); print(df.head(3))
print()
print("describe:"); print(df['Flow_m3s'].describe())
print()
# Boolean condition on a column: True where Flag == 'GOOD'
print(f"GOOD records   : {(df['Flag']=='GOOD').sum()}")
print(f"MISSING records: {(df['Flag']=='MISSING').sum()}")

### 🔁 Try this

Filter the DataFrame to show only GOOD records with flow > 400 m³/s:

`df[(df['Flag']=='GOOD') & (df['Flow_m3s'] > 400)]`

How many such records are there?

---
## Session Summary — Pandas Fundamentals

| Object / Method | What it does | Example |
|---|---|---|
| `pd.Series(values, index, name)` | 1-D labelled array | Monthly rainfall with month names |
| `pd.DataFrame(dict)` | 2-D labelled table from dictionary | Station data table |
| `pd.read_csv(file)` | Load CSV into DataFrame | `pd.read_csv('cwc.csv')` |
| `.sum()` | Sum of all values | `monthly_rain.sum()` |
| `.max()` / `.min()` | Maximum / minimum value | `df['Flow'].max()` |
| `.idxmax()` / `.idxmin()` | **Label** of max / min | `monthly_rain.idxmax()` → `'Jun'` |
| `series[[list of labels]]` | Select multiple by label | `rain[['Jun','Jul']]` |
| `df.shape` | `(rows, cols)` tuple | `(5, 5)` |
| `df.dtypes` | Data type per column | `float64`, `object` |
| `df.head(n)` | First n rows | `df.head(3)` |
| `df.describe()` | Statistical summary of numeric cols | count, mean, std, percentiles |
| `(df['col']=='value').sum()` | Count matching rows | `(df['Flag']=='GOOD').sum()` |

---
## Day 28 Assignment

```python
data = {'Station':['Chennai','Mumbai','Kolkata','Delhi','Hyderabad','Bengaluru'],
        'State':['Tamil Nadu','Maharashtra','West Bengal','Delhi','Telangana','Karnataka'],
        'Annual_mm':[1400,2167,1582,714,812,967],
        'Monsoon_mm':[890,1850,1246,572,620,721],
        'Elevation_m':[6,11,9,216,536,920]}
```

1. Create a DataFrame from this dictionary
2. Add a column `Monsoon_pct = Monsoon_mm / Annual_mm * 100`
3. Find the station with the highest monsoon percentage using `.idxmax()`
4. Filter to show only stations above 200 m elevation

### ▶ Assignment cell

In [ ]:
import pandas as pd

data = {'Station':['Chennai','Mumbai','Kolkata','Delhi','Hyderabad','Bengaluru'],
        'State':['Tamil Nadu','Maharashtra','West Bengal','Delhi','Telangana','Karnataka'],
        'Annual_mm':[1400,2167,1582,714,812,967],
        'Monsoon_mm':[890,1850,1246,572,620,721],
        'Elevation_m':[6,11,9,216,536,920]}
df = pd.DataFrame(data)

df['Monsoon_pct'] = ???
wettest = ???
high_elev = ???

print(df)
print(f"Wettest monsoon: {wettest}")
print(high_elev)

---
- [ ] Run all cells — verify outputs match expected outputs above
- [ ] Complete the assignment cell (replace `???` placeholders)
- [ ] Upload to GitHub: `Unit4_Pandas/CE541E08_U4_Day28.ipynb`
- [ ] Commit message: `Day 28 assignment completed`

*CE541E08 · Civil Engineering · Christ University · 2026-27 · Dr. Arpan Pradhan*